In [ ]:
import os
from openai import OpenAI
from pypdf import PdfReader
import sqlite3 as sqlite3
import json as json
import re as re
from dotenv import load_dotenv

In [ ]:
load_dotenv()
api_key = os.getenv("API_KEY")
base_url  = os.getenv("BASE_URL")

In [ ]:
client  =OpenAI(api_key=api_key,
                base_url=base_url)

In [ ]:
# reading the file and extract each pages words into text

path  = "PDF.pdf"
text = " "
reader = PdfReader(path)

for i in range(len(reader.pages)):
    text += reader.pages[i].extract_text()

In [ ]:
# devide the whole text into chunck

def chunk_text(text, chunk_size=500, overlap=100):
    chunks = []

    step = chunk_size - overlap

    for i in range(0, len(text), step):
        chunk = text[i:i + chunk_size]
        chunks.append(chunk)

    return chunks
chunks = chunk_text(text,1500,overlap=250)

In [ ]:
# name of the model you wanna load
model = "inclusionai/ling-3.0-flash-fin:free"

In [ ]:
# your request to the model 
response = client.chat.completions.create(
    model=model, 
    messages=[
        {   "role": "system",
            "content": """ your are an text extractor assisstant
            and bring the data mentioned in a valid json style only, and no more information
            name of author :author
            name of professor :professor
            name of university:university
            title of the proposal in persian only :title
             """},
        {
            "role": "user",
            "content": chunks[0] }])
print(response.choices[0].message.content)

In [ ]:
# a regular expression is used to convert the answer to a json
answer = response.choices[0].message.content

match = re.search(r'\{.*\}', answer, re.DOTALL)

if match:
    data = json.loads(match.group(0))
    print(data)

In [ ]:
# the data extracted from PDF is inserted to a sqlite database
conn = sqlite3.connect('proposals.db')
cursor = conn.cursor()
cursor.execute('''
    CREATE TABLE IF NOT EXISTS proposal (
        author_name TEXT,
        university_name TEXT,
        professor_name TEXT,
        proposal_title TEXT)''')
cursor.execute('''
    INSERT INTO proposal (author_name, university_name, professor_name, proposal_title)
    VALUES (?, ?, ?, ?)''', (data["author"], data["university"], data["professor"], data["title"]))

conn.commit()
conn.close()